# This notebook is meant to pool the dataasets eICU, MIMIC-III and MIMIC-IV in pairs of two used for the self-supervised pre-training extension of YAIB where one fine-tuning test set is held-out

### Imports

In [1]:
import os
import copy
import logging

import gin
import json
import hashlib
import pandas as pd
import polars as pl
from pathlib import Path
import pickle
from timeit import default_timer as timer
from sklearn.model_selection import StratifiedKFold, KFold, StratifiedShuffleSplit, ShuffleSplit
from icu_benchmarks.data.preprocessor import Preprocessor, PandasClassificationPreprocessor, PolarsClassificationPreprocessor
from icu_benchmarks.constants import RunMode
from icu_benchmarks.run_utils import check_required_keys
from icu_benchmarks.data.constants import DataSplit as Split, DataSegment as Segment, VarType as Var

from icu_benchmarks.data.split_process_data import *
from icu_benchmarks.cross_validation import execute_repeated_cv  # adjust if path is different
from icu_benchmarks.run import *


### Pool datasets

In [2]:
vars_dict = {
    "GROUP": "stay_id",
    "SEQUENCE": "time",
    "LABEL": "label",
    "DYNAMIC": ["alb", "alp", "alt", "ast", "be", "bicar", "bili", "bili_dir", "bnd", "bun", "ca", "cai", "ck", "ckmb", "cl",
        "crea", "crp", "dbp", "fgn", "fio2", "glu", "hgb", "hr", "inr_pt", "k", "lact", "lymph", "map", "mch", "mchc", "mcv",
        "methb", "mg", "na", "neut", "o2sat", "pco2", "ph", "phos", "plt", "po2", "ptt", "resp", "sbp", "temp", "tnt", "urine",
        "wbc"],
    "STATIC": ["age", "sex", "height", "weight"],
}

# Load the gin config
gin.parse_config_file("/work3/s185395/YAIB/configs/tasks/Regression.gin")


/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


ParsedConfigFileIncludesAndImports(filename='/work3/s185395/YAIB/configs/tasks/Regression.gin', imports=[], includes=[ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/Imports.gin', imports=['icu_benchmarks.data.split_process_data', 'icu_benchmarks.data.loader', 'icu_benchmarks.models.wrappers', 'icu_benchmarks.models.dl_models', 'icu_benchmarks.models.ml_models'], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/PredictionTaskVariables.gin', imports=[], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/CrossValidation.gin', imports=[], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/Dataloader.gin', imports=[], includes=[])])

In [3]:
# Define data directory 
data_dir = "/work3/s185395/YAIB-cohorts/data"

# Call the preprocessing function with the new scale
mimic = preprocess_data(
    data_dir=Path(f"{data_dir}/los/mimic"),
    seed=2222,
    generate_cache=True,
    load_cache=False,
    debug=False,
    use_static = True,
    runmode=RunMode.regression
)

# Call the preprocessing function with the new scale
miiv = preprocess_data(
    data_dir=Path(f"{data_dir}/los/miiv"),
    seed=2222,
    generate_cache=True,
    load_cache=False,
    debug=False,
    use_static = True,
    runmode=RunMode.regression
)

# Call the preprocessing function with the new scale
eicu = preprocess_data(
    data_dir=Path(f"{data_dir}/los/eicu"),
    seed=2222,
    generate_cache=True,
    load_cache=False,
    debug=False,
    use_static = True,
    runmode=RunMode.regression
)


# Tweeking of pooling function from pooling.py provided by YAIB

In [ ]:
from pathlib import Path
import logging
import pandas as pd
from sklearn.model_selection import train_test_split
from icu_benchmarks.data.constants import DataSegment as Segment, VarType as Var
from icu_benchmarks.constants import RunMode
import pyarrow.parquet as pq

class PooledDataset:
    hirid_eicu_miiv = ["hirid", "eicu", "miiv"]
    aumc_hirid_eicu = ["aumc", "hirid", "eicu"]
    aumc_eicu_miiv = ["aumc", "eicu", "miiv"]
    aumc_hirid_miiv = ["aumc", "hirid", "miiv"]
    aumc_hirid_eicu_miiv = ["aumc", "hirid", "eicu", "miiv"]
    eicu_miiv = ["eicu", "miiv"]
    eicu_mimic = ["eicu", "mimic"]
    mimic_miiv = ["miiv", "mimic"]
    eicu_mimic_miiv = ["eicu", "mimic", "miiv"]


class PooledData:
    def __init__(
        self,
        data_dir,
        vars_dict,
        datasets,
        file_names,
        shuffle=False,
        stratify=None,
        runmode=RunMode.classification,
        save_test=True,
    ):
        self.data_dir = data_dir
        self.vars_dict = vars_dict  
        self.datasets = datasets
        self.file_names = file_names
        self.shuffle = shuffle
        self.stratify = stratify
        self.runmode = runmode
        self.save_test = save_test

    def generate(self, datasets, samples=10000, seed=42):
        data = {}
        for folder in self.data_dir.iterdir():
            if folder.is_dir() and folder.name in datasets:
                data[folder.name] = {
                    f: pq.read_table(folder / self.file_names[f]).to_pandas(self_destruct=True)
                    for f in self.file_names
                }
        data = self._pool_datasets(
            datasets=data,
            samples=samples,
            vars_dict=self.vars_dict,  
            shuffle=self.shuffle,
            seed=seed,
            runmode=self.runmode,
            data_dir=self.data_dir,
            save_test=self.save_test,
        )
        self._save_pooled_data(self.data_dir, data, datasets, self.file_names, samples=samples)

    def _save_pooled_data(self, data_dir, data, datasets, file_names, samples=10000):
        save_folder = "_".join(datasets) + f"_{samples}"
        save_dir = data_dir / save_folder
        if not save_dir.exists():
            save_dir.mkdir()
        for key, value in data.items():
            value.to_parquet(save_dir / Path(file_names[key]))
        logging.info(f"Saved pooled data at {save_dir}")

    def _pool_datasets(
        self,
        datasets=None,
        samples=10000,
        vars_dict=None,
        seed=42,
        shuffle=True,
        runmode=RunMode.classification,
        data_dir=Path("data"),
        save_test=True,
    ):
        if datasets is None:
            datasets = {}
        if vars_dict is None:
            vars_dict = {}
        if len(datasets) == 0:
            raise ValueError("No datasets supplied.")

        pooled_data = {
            Segment.static.lower(): [],
            Segment.dynamic.lower(): [],
            Segment.outcome.lower(): []
        }

        id_col = vars_dict[Var.group]
        int_id = 0

        for key, value in datasets.items():
            int_id += 1
            repeated_digit = str(int_id) * 4

            outcome_full = value[Segment.outcome.lower()]
            static_full = value[Segment.static.lower()]
            dynamic_full = value[Segment.dynamic.lower()]

            stays = pd.Series(outcome_full[id_col].unique())
            print(f"[{key}] Total stays: {len(stays)}")

            if samples is None:
                train_ids = stays
                test_ids = pd.Series([], dtype=stays.dtype)
                print(f"[{key}] Using all {len(train_ids)} stays (no test split)")
            else:
                if runmode is RunMode.classification:
                    labels = outcome_full.groupby(id_col).max()[vars_dict[Var.label]].reset_index(drop=True)
                    train_ids, test_ids = train_test_split(
                        stays, stratify=labels, shuffle=shuffle, random_state=seed, train_size=samples
                    )
                else:
                    train_ids, test_ids = train_test_split(
                        stays, shuffle=shuffle, random_state=seed, train_size=samples
                    )

            # Handle test set first, only if sampling
            if save_test and samples is not None:
                outcome_test, static_test, dynamic_test = self._select_stays(
                    outcome_full.copy(), static_full.copy(), dynamic_full.copy(),
                    test_ids, id_col, repeated_digit
                )
                save_folder = key + f"_test_{len(test_ids)}"
                save_dir = data_dir / save_folder
                save_dir.mkdir(exist_ok=True)
                outcome_test.to_parquet(save_dir / Path("outc.parquet"))
                static_test.to_parquet(save_dir / Path("sta.parquet"))
                dynamic_test.to_parquet(save_dir / Path("dyn.parquet"))
                logging.info(f"Saved test data at {save_dir}")

            # Now handle training stays
            outcome_train, static_train, dynamic_train = self._select_stays(
                outcome_full.copy(), static_full.copy(), dynamic_full.copy(),
                train_ids, id_col, repeated_digit
            )

            pooled_data[Segment.static.lower()].append(static_train)
            pooled_data[Segment.dynamic.lower()].append(dynamic_train)
            pooled_data[Segment.outcome.lower()].append(outcome_train)

        for key in pooled_data:
            pooled_data[key] = pd.concat(pooled_data[key], ignore_index=True)

        return pooled_data


    def _select_stays(self, outcome, static, dynamic, select, id_col, repeated_digit=1):
        print(f"[Select] Matching {len(select)} stays in outcome shape {outcome.shape}")
        matched = outcome[outcome[id_col].isin(select)]
        print(f"[Select] Found {len(matched)} matching rows before ID modification")
        
        outcome = outcome.loc[outcome[id_col].isin(select)]
        static = static.loc[static[id_col].isin(select)]
        dynamic = dynamic.loc[dynamic[id_col].isin(select)]

        outcome[id_col] = outcome[id_col].map(lambda x: int(str(x) + repeated_digit))
        static[id_col] = static[id_col].map(lambda x: int(str(x) + repeated_digit))
        dynamic[id_col] = dynamic[id_col].map(lambda x: int(str(x) + repeated_digit))
        return outcome, static, dynamic


In [ ]:
from pathlib import Path
#from icu_benchmarks.data.pooling import PooledData, PooledDataset
from icu_benchmarks.constants import RunMode
import logging

logging.basicConfig(level=logging.INFO)

file_names = {
    "outcome": "outc.parquet",
    "static": "sta.parquet",
    "dynamic": "dyn.parquet"
}

def generate_pooled_data(data_dir, vars_dict, datasets, file_names, seed, runmode):
    pooled = PooledData(
        data_dir=data_dir,
        vars_dict=vars_dict,  
        datasets=datasets,
        file_names=file_names,
        runmode=runmode,
    )
    pooled.generate(datasets=datasets, samples=None, seed=seed)


# Uncomment based on which datasets you want to pool
#Datasets = [PooledDataset.eicu_miiv]
#Datasets = [PooledDataset.eicu_mimic]
#Datasets = [PooledDataset.mimic_miiv]
Datasets = [PooledDataset.eicu_mimic_miiv]


seed = 42
runmode = RunMode.regression

for item in Datasets:
    generate_pooled_data(
        Path(f"{data_dir}/los"),
        vars_dict=vars_dict,
        datasets=item,
        file_names=file_names,
        seed=seed,
        runmode=runmode
    )